In [23]:
from pathlib import Path
import sys
import os

ROOT = Path(os.getcwd())

# if notebook is in notebooks/, go up one level
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA = ROOT / "data"

sys.path.append(str(ROOT))

In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

## Biomass-producing reactions dataset

## Macromolecular composition dataset

## Machine Learning

* split train/test
* make features

In [25]:
from sklearn.linear_model import LinearRegression

**SPLIT**

In [26]:
data = pd.read_pickle(DATA/'ml_datasets/all_raw_samples.pkl')

In [27]:
train_set, test_set = train_test_split(data.index, test_size=0.2, stratify=data['source'], random_state=42)

train_data = data.loc[train_set]
test_data = data.loc[test_set]

### Feature matrix

In [28]:
ex_cols = [col for col in data.columns if col.startswith('EX_')]
bm_cols = [col for col in data.columns if 'mass' in col]
rna_cols = data['mRNA_biomass_to_biomass'] + data['rRNA_biomass_to_biomass'] + data['tRNA_biomass_to_biomass']

**Compute ratios**

In [31]:
ratio_train = train_data[bm_cols]
ratio_test = test_data[bm_cols]

########## TRAIN ###########################
ratio_train.drop(columns=['dummy_protein_to_mass', 'peptidoglycan_biomass_to_biomass', 'constituent_biomass_to_biomass'], inplace=True)
ratio_train.drop(columns=[col for col in ratio_train.columns if 'RNA' in col and 'ncRNA' not in col], inplace=True)
ratio_train.insert(1, 'RNA_biomass_to_biomass', rna_cols)

########## TEST ####################################
ratio_test.drop(columns=['dummy_protein_to_mass', 'peptidoglycan_biomass_to_biomass', 'constituent_biomass_to_biomass'], inplace=True)
ratio_test.drop(columns=[col for col in ratio_test.columns if 'RNA' in col and 'ncRNA' not in col], inplace=True)
ratio_test.insert(1, 'RNA_biomass_to_biomass', rna_cols)


In [32]:
biomass_train = ratio_train.copy()
biomass_test = ratio_test.copy()

In [38]:
ratio_train['ncRNA/RNA'] = ratio_train['ncRNA_biomass_to_biomass']/ratio_train['RNA_biomass_to_biomass']
ratio_train['protein/DNA'] = ratio_train['protein_biomass_to_biomass']/ratio_train['DNA_biomass_to_biomass']
ratio_train['RNA/DNA'] = ratio_train['RNA_biomass_to_biomass']/ratio_train['DNA_biomass_to_biomass']
ratio_train['ncRNA/DNA'] = ratio_train['ncRNA_biomass_to_biomass']/ratio_train['DNA_biomass_to_biomass']


ratio_test['ncRNA/RNA'] = ratio_test['ncRNA_biomass_to_biomass']/ratio_test['RNA_biomass_to_biomass']
ratio_test['protein/DNA'] = ratio_test['protein_biomass_to_biomass']/ratio_test['DNA_biomass_to_biomass']
ratio_test['RNA/DNA'] = ratio_test['RNA_biomass_to_biomass']/ratio_test['DNA_biomass_to_biomass']
ratio_test['ncRNA/DNA'] = ratio_test['ncRNA_biomass_to_biomass']/ratio_test['DNA_biomass_to_biomass']

**LOG-TRANSFORM**

In [40]:
log_train = ratio_train.copy()
log_train = np.log(log_train)

log_test = ratio_test.copy()
log_test = np.log(log_test)

In [42]:
train_features = log_train.drop(columns='lipid_biomass_to_biomass')
test_features = log_test.drop(columns='lipid_biomass_to_biomass')

**L2-NORM**

In [47]:
l2_train = train_features.iloc[:, :5]
l2_test = test_features.iloc[:, :5]

In [49]:
from sklearn.preprocessing import normalize

train_transformed =  pd.DataFrame(
    normalize(l2_train.values, norm='l2', axis=1),
    index=l2_train.index,
    columns=l2_train.columns
)

In [52]:
test_transformed = pd.DataFrame(
    normalize(l2_test.values, norm='l2', axis=1),
    index=l2_test.index,
    columns=l2_test.columns
)

In [55]:
train_transformed.columns.tolist()

['protein_biomass_to_biomass',
 'RNA_biomass_to_biomass',
 'ncRNA_biomass_to_biomass',
 'DNA_biomass_to_biomass',
 'prosthetic_group_biomass_to_biomass']

**Feature DataFrames**

In [54]:
log_train.columns.tolist()

['protein_biomass_to_biomass',
 'RNA_biomass_to_biomass',
 'ncRNA_biomass_to_biomass',
 'DNA_biomass_to_biomass',
 'lipid_biomass_to_biomass',
 'prosthetic_group_biomass_to_biomass',
 'biomass_dilution',
 'ncRNA/RNA',
 'protein/DNA',
 'RNA/DNA',
 'ncRNA/DNA']

In [56]:
train_df = train_transformed.copy()
test_df = test_transformed.copy()

train_df[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']] = log_train[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']]
test_df[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']] = log_test[['ncRNA/RNA', 'protein/DNA','RNA/DNA','ncRNA/DNA', 'biomass_dilution']]

In [68]:
train_df[ex_cols] = train_data[ex_cols]
train_df[['source', 'outlier', 'o2_e_shadow', 'source_shadow']] = train_data[['source', 'outlier', 'o2_e_shadow', 'source_shadow']]

test_df[ex_cols] = test_data[ex_cols]
test_df[['source', 'outlier', 'o2_e_shadow', 'source_shadow']] = test_data[['source', 'outlier', 'o2_e_shadow', 'source_shadow']]


In [ ]:
# train_df.to_pickle('L2_TRAIN.pkl')
# test_df.to_pickle('L2_IID_TEST.pkl')

In [76]:
raw_train = train_features.copy()
raw_test = test_features.copy()

raw_train[ex_cols] = train_data[ex_cols]
raw_test[ex_cols] = test_data[ex_cols]


raw_train[['source', 'outlier', 'o2_e_shadow', 'source_shadow']] = train_data[['source', 'outlier', 'o2_e_shadow', 'source_shadow']]
raw_test[['source', 'outlier', 'o2_e_shadow', 'source_shadow']] = test_data[['source', 'outlier', 'o2_e_shadow', 'source_shadow']]


In [ ]:
# raw_train.to_pickle('LOG_TRAIN.pkl')
# raw_test.to_pickle('LOG_IID_TEST.pkl')